# How an LLM Works, From the Inside

A hands-on tour of a transformer's guts — built tiny, from scratch, and trained on real data.

## The core idea

A neural network is a large collection of numerical values — its **parameters** — organised
into a sequence of functions called **layers**. *Training* is the process of iteratively
adjusting those parameters so that the layers, applied in order, transform an input into a
useful output.

This notebook builds the smallest *honest* version of a modern language model — a
**transformer** — so you can see every one of those numbers and what it does. Instead of
hand-waving, we'll load a real dataset, tokenize it, embed it, run it through attention and
stacked layers, count every parameter by hand, train the thing, and measure it.

Our task: classify the sentiment of real tweets as **negative**, **neutral**, or **positive**.
The machine we build here is the same one inside ChatGPT — just tiny, and pointed at an
easier job. Everything runs on a plain CPU in a couple of minutes.

In [ ]:
# On Colab these are mostly preinstalled; this is a quiet guard for local runs.
import importlib.util, subprocess, sys
for pkg in ["torch", "datasets", "scikit-learn", "matplotlib"]:
    name = "sklearn" if pkg == "scikit-learn" else pkg
    if importlib.util.find_spec(name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("dependencies ready")

In [ ]:
import random, re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Hyperparameters (all tunables live here) ---
BLOCK_SIZE   = 32           # max tokens per tweet
D_MODEL      = 64           # embedding / hidden width
N_HEADS      = 2            # attention heads per block
N_LAYERS     = 2            # stacked transformer blocks
D_FF         = 4 * D_MODEL  # feed-forward inner width
DROPOUT      = 0.2
N_CLASSES    = 3            # negative / neutral / positive

# --- Training / data budget (kept small so it runs on a CPU in ~1-2 min) ---
SUBSET       = 15000        # tweets pulled from the corpus, then split 80/10/10
BATCH_SIZE   = 64
EPOCHS       = 8
LR           = 1e-3
WEIGHT_DECAY = 0.1          # mild regularization to fight overfitting

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. The data: real tweets, three sentiments

We use [`tweet_eval`](https://huggingface.co/datasets/cardiffnlp/tweet_eval) (the `sentiment` subset, from
SemEval-2017) — thousands of real tweets, each labelled `0 = negative`, `1 = neutral`, or
`2 = positive`.

A crucial habit in ML: split your data into three parts, each with a different job.

| split | job |
|-------|-----|
| **train** | the model learns from these |
| **validation** | we watch this during training to catch overfitting and decide when to stop |
| **test** | touched **once**, at the very end, for one honest score |

We build these splits ourselves with a **stratified** split — meaning each split keeps the same
class balance — so the three are comparable. (`tweet_eval` does ship its own splits, but its
official *test* set was sampled from a different time period and has a noticeably different class
balance; mixing distributions would muddy the lesson, so here we draw all three splits from one
pool.)

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

raw = load_dataset("cardiffnlp/tweet_eval", "sentiment")
LABELS = ["negative", "neutral", "positive"]

pool = raw["train"].shuffle(seed=SEED).select(range(min(SUBSET, len(raw["train"]))))
texts, labels = list(pool["text"]), list(pool["label"])

# stratified 80 / 10 / 10 split (same class balance in each part)
train_texts, tmp_t, train_labels, tmp_y = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    tmp_t, tmp_y, test_size=0.5, random_state=SEED, stratify=tmp_y)

print(f"train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}")
for t, y in list(zip(train_texts, train_labels))[:3]:
    print(f"[{LABELS[y]:8}] {t[:80]}")

In [ ]:
counts = Counter(train_labels)
print("train class counts:", {LABELS[k]: counts[k] for k in range(N_CLASSES)})

majority_class = counts.most_common(1)[0][0]
baseline_acc = float(np.mean(np.array(test_labels) == majority_class))
print(f"majority class = {LABELS[majority_class]!r}; "
      f"always-guess-majority test accuracy = {baseline_acc:.3f}")
assert len(train_texts) > 0 and len(test_texts) > 0

The classes are imbalanced (lots of *neutral*), so "always guess the most common class"
already scores the accuracy printed above. **Our model has to beat that** to have learned
anything real — keep that number in mind.

## 2. Tokenization: turning text into integers

A model can only process numbers, never raw text. **Tokenization** splits text into pieces
(here, whole words) and maps each to an integer id. Big models use *subword* tokens; we use
plain words for clarity — one word, one token.

Two special tokens earn their keep:

- **`<pad>`** — tweets have different lengths, but a tensor is a rectangle. We pad short tweets
  to a fixed length with this filler token.
- **`<unk>`** — at test time we'll meet words that never appeared in training. They all map to
  this single "unknown" token.

One rule we must respect: **the vocabulary is built from the *training* split only.** Peeking at
validation/test to build the vocab would leak information — a subtle but real form of cheating.

In [ ]:
def tokenize(text):
    # lowercase, keep words / @handles / #hashtags as single tokens
    return re.findall(r"[a-z0-9@#']+", text.lower())

PAD, UNK = "<pad>", "<unk>"
counter = Counter(tok for t in train_texts for tok in tokenize(t))
vocab = [PAD, UNK] + [w for w, _ in counter.most_common()]
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
VOCAB_SIZE = len(vocab)

print(f"vocab size = {VOCAB_SIZE}")
print("first 12 tokens:", vocab[:12])

In [ ]:
def encode(text):
    ids = [stoi.get(tok, stoi[UNK]) for tok in tokenize(text)][:BLOCK_SIZE]
    ids = ids + [stoi[PAD]] * (BLOCK_SIZE - len(ids))
    return ids

def make_tensors(texts, labels):
    X = torch.tensor([encode(t) for t in texts], dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y

Xtr, ytr = make_tensors(train_texts, train_labels)
Xval, yval = make_tensors(val_texts, val_labels)
Xte, yte = make_tensors(test_texts, test_labels)

example = train_texts[0]
print("text  :", example[:70])
print("tokens:", tokenize(example)[:12])
print("ids   :", encode(example)[:12])
print("X shape:", tuple(Xtr.shape), "(rows = tweets, cols = token positions)")
assert Xtr.shape == (len(train_texts), BLOCK_SIZE)

## 3. Embeddings: turning integers into vectors ("vectoring")

An integer id like `42` carries no meaning — `42` isn't "more" than `7` in any useful sense.
So the first real layer is an **embedding table**: a big matrix of shape
`(VOCAB_SIZE, D_MODEL)` with **one learnable row of numbers per token**.

Looking up a token = grabbing its row. Those rows *are* parameters: training reshapes them so
that words used in similar ways drift to similar vectors. This step — symbols becoming
vectors — is what people loosely call **"vectoring"** or *embedding*.

In [ ]:
tok_emb = nn.Embedding(VOCAB_SIZE, D_MODEL)
print("embedding table shape:", tuple(tok_emb.weight.shape), "<- (vocab, d_model)")
print("parameters in this ONE table:", tok_emb.weight.numel())

ids = Xtr[:1]            # one tweet: (1, BLOCK_SIZE)
vecs = tok_emb(ids)      # (1, BLOCK_SIZE, D_MODEL)
print("one tweet, embedded:", tuple(vecs.shape), f"<- each token is now a {D_MODEL}-d vector")
assert vecs.shape == (1, BLOCK_SIZE, D_MODEL)

## 4. Positional encoding: putting word order back

Here's a surprise: the attention mechanism we're about to build is **order-blind**. It treats
a tweet as an unordered *bag* of token vectors — "dog bites man" and "man bites dog" would look
identical. That's clearly wrong for language.

The fix: a second small table indexed by **position** (0, 1, 2, …). We add the position-vector
to each token-vector, stamping "I am the 3rd word" into the representation.

In [ ]:
pos_emb = nn.Embedding(BLOCK_SIZE, D_MODEL)
positions = torch.arange(BLOCK_SIZE)         # 0, 1, 2, ...
x = tok_emb(ids) + pos_emb(positions)        # broadcast-add position onto each token
print("token + position:", tuple(x.shape))
assert x.shape == (1, BLOCK_SIZE, D_MODEL)

## 5. Self-attention: the heart of the transformer

This is the idea that made modern LLMs work. Each token gets to **look at every other token**
and pull in what's relevant. Concretely, every token emits three vectors:

- a **query** *q* — "what am I looking for?"
- a **key** *k* — "what do I offer?"
- a **value** *v* — "what I'll hand over if you attend to me"

For a pair of tokens, their relevance is the dot product *q · k* (big = aligned). We scale by
`1/√d` to keep numbers tame, softmax across all tokens so the weights sum to 1, then take the
weighted sum of the **values**. In one line:

$$\text{attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

Let's run exactly that on our one embedded tweet and watch the shapes.

In [ ]:
d_head = D_MODEL  # a single head for this illustration
Wq, Wk, Wv = (nn.Linear(D_MODEL, d_head, bias=False) for _ in range(3))

q, k, v = Wq(x), Wk(x), Wv(x)                    # each (1, T, d_head)
scores  = q @ k.transpose(-2, -1) / d_head**0.5  # (1, T, T): token-to-token similarity
weights = scores.softmax(dim=-1)                 # each row sums to 1
out     = weights @ v                            # (1, T, d_head): weighted blend of values

print("q, k, v       :", tuple(q.shape))
print("scores (T x T):", tuple(scores.shape), "<- how much each token attends to each other")
print("weight row sums:", [round(s, 3) for s in weights[0].sum(-1)[:5].tolist()], "(~1.0)")
print("attention out :", tuple(out.shape))
assert torch.allclose(weights.sum(-1), torch.ones_like(weights.sum(-1)), atol=1e-5)
assert out.shape == (1, BLOCK_SIZE, d_head)

Real transformers run several of these in parallel — **multi-head attention** — so different
heads can specialise (one tracks syntax, another sentiment, …). Each head works on a slice of
the vector; the results are concatenated back together. The next section packages this into a
reusable module.

## 6. From one head to a layer, and a whole model

A transformer **block** (one "layer") stacks two sub-pieces, each wrapped in a **residual
connection** (`x = x + sublayer(x)`, so information can skip ahead) and **LayerNorm** (which
keeps activations well-scaled):

1. **multi-head self-attention** — tokens mix information across positions
2. a **feed-forward MLP** — each token is transformed on its own

The full model: embed tokens + positions → stack `N_LAYERS` blocks → **mean-pool** over the
real (non-pad) tokens into one vector per tweet → a linear **head** producing 3 scores
(**logits**), one per sentiment class.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        def heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = heads(q), heads(k), heads(v)
        att = (q @ k.transpose(-2, -1) / self.d_head**0.5).softmax(-1)
        att = self.drop(att)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual around attention
        x = x + self.mlp(self.ln2(x))    # residual around the MLP
        return x


class TinyTransformerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=stoi[PAD])
        self.pos_emb = nn.Embedding(BLOCK_SIZE, D_MODEL)
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.ModuleList([Block(D_MODEL, N_HEADS, D_FF, DROPOUT)
                                     for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, N_CLASSES)

    def forward(self, idx):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        mask = (idx != stoi[PAD]).unsqueeze(-1).float()       # ignore pad in the average
        pooled = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.head(pooled)                              # (B, N_CLASSES) logits


model = TinyTransformerClassifier().to(DEVICE)
logits = model(Xtr[:4].to(DEVICE))
print("logits shape:", tuple(logits.shape), "<- (batch, 3 class scores)")
assert logits.shape == (4, N_CLASSES)

### Counting every parameter — and reconciling it by hand

"This model has X million parameters" stops being mysterious once you can add them up yourself.
Let's break the total down by component and check our hand-sum equals what PyTorch reports.

In [ ]:
def numel(m):
    return sum(p.numel() for p in m.parameters())

total  = numel(model)
emb    = model.tok_emb.weight.numel() + model.pos_emb.weight.numel()
blocks = sum(numel(b) for b in model.blocks)
head   = numel(model.head) + numel(model.ln_f)

print(f"{'embeddings (token+pos)':28} {emb:>9,}")
print(f"{'transformer blocks':28} {blocks:>9,}")
print(f"{'final norm + head':28} {head:>9,}")
print(f"{'-'*38}")
print(f"{'hand-summed total':28} {emb + blocks + head:>9,}")
print(f"{'model.parameters() total':28} {total:>9,}")
assert emb + blocks + head == total, "hand breakdown must equal the real count!"

print(f"\nThe token embedding table alone is "
      f"{model.tok_emb.weight.numel() / total:.0%} of all parameters.")

## 7. Before training: just random numbers

Right now every parameter is random, so the model should perform at roughly chance
(1/3 for three classes). We measure that now to have a "before" picture — and we grab a single
weight so we can literally watch it change once training starts.

In [ ]:
@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    preds = model(X.to(DEVICE)).argmax(1).cpu()
    return (preds == y).float().mean().item()

print(f"untrained val accuracy: {accuracy(model, Xval, yval):.3f} "
      f"(chance ~ {1/N_CLASSES:.3f})")
watch_before = model.head.weight[0, 0].item()
print("a single head weight, before training:", watch_before)

## 8. Training: nudging the numbers

The loop is the whole of "learning":

1. **forward pass** — run a batch through the model to get logits
2. **loss** — cross-entropy measures how wrong the predictions are
3. **`.backward()`** — autograd computes, for every parameter, which way to nudge it
4. **optimizer step** — take a small step downhill for all parameters at once

We track **validation loss** alongside training loss: when train keeps dropping but validation
turns back up, the model is memorising (overfitting) rather than learning. To exploit that, we
use **early stopping** — we remember the model from the epoch with the *best* validation loss and
restore it at the end, instead of keeping the over-trained final version. `WEIGHT_DECAY` adds a
gentle pull toward smaller weights, another nudge against overfitting.

In [ ]:
import copy

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def iterate_batches(X, y, bs, shuffle=True):
    idx = torch.randperm(len(X)) if shuffle else torch.arange(len(X))
    for i in range(0, len(X), bs):
        j = idx[i:i + bs]
        yield X[j].to(DEVICE), y[j].to(DEVICE)

@torch.no_grad()
def eval_loss(model, X, y):
    model.eval()
    losses = [F.cross_entropy(model(xb), yb).item()
              for xb, yb in iterate_batches(X, y, BATCH_SIZE, shuffle=False)]
    return float(np.mean(losses))

train_hist, val_hist = [], []
best_val, best_state, best_epoch = float("inf"), None, 0
for epoch in range(EPOCHS):
    model.train()
    ep_losses = []
    for xb, yb in iterate_batches(Xtr, ytr, BATCH_SIZE):
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        ep_losses.append(loss.item())
    tr, vl = float(np.mean(ep_losses)), eval_loss(model, Xval, yval)
    train_hist.append(tr); val_hist.append(vl)
    if vl < best_val:                                  # remember the best-so-far model
        best_val, best_state, best_epoch = vl, copy.deepcopy(model.state_dict()), epoch + 1
    print(f"epoch {epoch + 1:2d}  train_loss={tr:.3f}  val_loss={vl:.3f}  "
          f"val_acc={accuracy(model, Xval, yval):.3f}")

model.load_state_dict(best_state)                      # early stopping: restore the best
print(f"\nrestored best model from epoch {best_epoch} (val_loss={best_val:.3f})")
assert min(val_hist) < val_hist[0], 'validation loss should drop -- the model should learn'

In [ ]:
watch_after = model.head.weight[0, 0].item()
print(f"same weight, after training: {watch_after:.5f} (was {watch_before:.5f}) "
      f"-- a parameter literally changed")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), train_hist, marker="o", label="train loss")
plt.plot(range(1, EPOCHS + 1), val_hist, marker="o", label="validation loss")
plt.axvline(best_epoch, color="gray", ls="--", label=f"best epoch ({best_epoch})")
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss")
plt.legend(); plt.title("Learning curve")
plt.tight_layout(); plt.show()

## 9. The honest score: the held-out test set

Now — and only now — we touch the **test** split, once. We report:

- **accuracy**, compared against the majority-class baseline from earlier, and
- **macro precision / recall / F1** — *macro* averages the score across the three classes
  equally, so the big "neutral" class can't paper over weak performance on the rarer ones.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

@torch.no_grad()
def predict(model, X):
    model.eval()
    return model(X.to(DEVICE)).argmax(1).cpu().numpy()

test_pred = predict(model, Xte)
test_true = yte.numpy()

test_acc = accuracy_score(test_true, test_pred)
print(f"test accuracy: {test_acc:.3f}   (majority baseline: {baseline_acc:.3f})")
print()
print(classification_report(test_true, test_pred, target_names=LABELS, digits=3))
assert test_acc >= baseline_acc, 'a trained model should beat always-guessing-majority'

In [ ]:
cm = confusion_matrix(test_true, test_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix (test)")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

### Why these metrics here — but not for ChatGPT

Precision, recall, F1 and accuracy all need a **discrete right answer** per example to score a
prediction as correct or not. Our classification task provides exactly that (each tweet has one
true label), so they apply cleanly.

A **generative** language model is different: it predicts a *probability distribution over the
next token*. There's no single right answer — many continuations are fine — so you can't tally
true/false positives. Those models are judged by the training **loss** and its intuitive cousin
**perplexity** (roughly, "how many tokens is the model effectively choosing between" — lower is
better). So the original instinct to reach for precision/recall fits *this* notebook precisely
because we framed it as classification.

## 10. Recap: the jargon, mapped to what you touched

| term | what it actually was, in this notebook |
|------|----------------------------------------|
| **parameter** | a single number in a weight matrix or embedding table — we counted them all and watched one change during training |
| **layer / block** | one `Block` (attention + MLP + residual + LayerNorm); the model stacked `N_LAYERS` of them |
| **embedding / "vectoring"** | the `tok_emb` lookup table turning token ids into vectors |
| **positional encoding** | the `pos_emb` table adding "where in the sentence" back in |
| **(self-)attention** | `softmax(QKᵀ/√d)·V` — every token blending in others by relevance |
| **logits** | the 3 raw class scores from `head`, before softmax turns them into probabilities |
| **training** | loss → `.backward()` → optimizer step, repeated |

A real LLM is *this exact machine*, scaled up: a wider `D_MODEL`, dozens of layers, subword
tokens, billions of parameters, and a head that predicts the next **token** instead of a
sentiment class — trained on a large fraction of the internet. But the moving parts are the
ones you just built and inspected by hand.